In [ ]:
import struct
from pathlib import Path

bin_path = Path('data/10302019.NASDAQ_ITCH50.bin')

print("=== ITCH Raw Byte Inspector ===")
print(f"File: {bin_path} ({bin_path.stat().st_size / 1e9:.1f} GB)\n")

with open(bin_path, 'rb') as f:
    # Read first 100 bytes (covers the first few messages)
    chunk = f.read(100)
    
    pos = 0
    print("Raw hex of first ~60 bytes (with labels):\n")
    
    while pos + 2 < len(chunk):
        length = (chunk[pos] << 8) | chunk[pos+1]
        msg_type = chr(chunk[pos+2])
        payload_start = pos + 3
        payload = chunk[payload_start : payload_start + length - 1]
        
        print(f"Position {pos:3d} → Length={length:3d}  Type='{msg_type}'")
        print("Raw bytes (hex):", ' '.join(f'{b:02X}' for b in chunk[pos:pos+2+length]))
        
        # Special case: show the first S message in detail
        if msg_type == 'S':
            print("\n" + "="*60)
            print("🔍 DETAILED BREAKDOWN OF FIRST 'S' MESSAGE (real data)")
            print("="*60)
            
            print(f"Byte 0-1 : Length     = {chunk[pos:pos+2].hex()}  → {length}")
            print(f"Byte 2   : Type       = {chunk[pos+2:pos+3]}     → 'S'")
            print(f"Byte 3-4 : stock_locate = {chunk[pos+3:pos+5].hex()} → 0 (always for S)")
            print(f"Byte 5-6 : tracking    = {chunk[pos+5:pos+7].hex()}")
            print(f"Byte 7-12: timestamp   = {chunk[pos+7:pos+13].hex()}")
            print(f"Byte 13  : event_code  = {chunk[pos+13:pos+14]}     → '{chr(chunk[pos+13])}'")
            
            # Unpack exactly like the notebook
            fmt = '>HH6sB'                                      # built from message_types.xlsx for S
            unpacked = struct.unpack(fmt, chunk[pos+3:pos+3+11])
            
            print("\nUnpacked tuple →", unpacked)
            print("event_code decoded →", chr(unpacked[3]))
            
            # Show mapping to your table
            print("\nMapping to message_types.xlsx:")
            print("stock_locate (offset 1, len 2) →", unpacked[0])
            print("tracking_number (offset 3, len 2) →", unpacked[1])
            print("timestamp (offset 5, len 6) → raw bytes above")
            print("event_code (offset 11, len 1) →", chr(unpacked[3]))
            
            break  # stop after first S message
        
        pos += 2 + length

print("\n✅ Done! You just looked inside the real raw .bin file 🎉")

=== ITCH Raw Byte Inspector ===
File: data/10302019.NASDAQ_ITCH50.bin (9.0 GB)

Raw hex of first ~60 bytes (with labels):

Position   0 → Length= 12  Type='S'
Raw bytes (hex): 00 0C 53 00 00 00 00 09 F5 E1 0B C9 1F 4F

🔍 DETAILED BREAKDOWN OF FIRST 'S' MESSAGE (real data)
Byte 0-1 : Length     = 000c  → 12
Byte 2   : Type       = b'S'     → 'S'
Byte 3-4 : stock_locate = 0000 → 0 (always for S)
Byte 5-6 : tracking    = 0000
Byte 7-12: timestamp   = 09f5e10bc91f
Byte 13  : event_code  = b'O'     → 'O'

Unpacked tuple → (0, 0, b'\t\xf5\xe1\x0b\xc9\x1f', 79)
event_code decoded → O

Mapping to message_types.xlsx:
stock_locate (offset 1, len 2) → 0
tracking_number (offset 3, len 2) → 0
timestamp (offset 5, len 6) → raw bytes above
event_code (offset 11, len 1) → O

✅ Done! You just looked inside the real raw .bin file 🎉


In [2]:
print("\n" + "="*80)
print("🔥 NEXT FEW MESSAGES — Real data from 10302019.NASDAQ_ITCH50.bin")
print("="*80)

with open(bin_path, 'rb') as f:
    f.seek(0)
    chunk = f.read(400)          # enough for first 6–8 messages
    
    pos = 0
    msg_count = 0
    
    while pos + 2 < len(chunk) and msg_count < 7:
        length = (chunk[pos] << 8) | chunk[pos + 1]
        if length == 0: break
        
        msg_type = chr(chunk[pos + 2])
        payload_start = pos + 3
        raw_message = chunk[pos : pos + 2 + length]
        
        print(f"\n📍 Message {msg_count+1:2d} | Start byte {pos:4d} | Length={length:3d} | Type='{msg_type}'")
        print("Raw hex →", ' '.join(f'{b:02X}' for b in raw_message))
        
        # Quick human label
        labels = {'S':'System Event', 'R':'Stock Directory', 'H':'Trading Action', 
                  'A':'Add Order', 'F':'Add Order + MPID', 'P':'Trade'}
        print("          →", labels.get(msg_type, "Other"))
        
        # Special nice breakdown for S and first R
        if msg_type == 'S' and msg_count == 0:
            print("   → First S (Start of Messages - O) ← as you saw")
        elif msg_type == 'S':
            event = chr(chunk[pos+13])
            ts_hex = chunk[pos+7:pos+13].hex()
            print(f"   → Event Code = '{event}' | Timestamp hex = {ts_hex}")
        
        elif msg_type == 'R' and msg_count == 2:   # usually the first R appears early
            stock = chunk[pos+11:pos+19].decode('ascii', errors='ignore').strip()
            print(f"   → First Stock Directory! Symbol = '{stock}'")
        
        pos += 2 + length
        msg_count += 1

print("\n✅ You just inspected the real raw ITCH feed!")
print("   Messages shown: System Event → System Event → Stock Directory → ...")
print("   This is exactly how the scanner in the notebook works!")


🔥 NEXT FEW MESSAGES — Real data from 10302019.NASDAQ_ITCH50.bin

📍 Message  1 | Start byte    0 | Length= 12 | Type='S'
Raw hex → 00 0C 53 00 00 00 00 09 F5 E1 0B C9 1F 4F
          → System Event
   → First S (Start of Messages - O) ← as you saw

📍 Message  2 | Start byte   14 | Length= 39 | Type='R'
Raw hex → 00 27 52 00 01 00 00 0A 35 A9 57 D3 60 41 20 20 20 20 20 20 20 4E 20 00 00 00 64 4E 43 5A 20 50 4E 20 31 4E 00 00 00 00 4E
          → Stock Directory

📍 Message  3 | Start byte   55 | Length= 39 | Type='R'
Raw hex → 00 27 52 00 02 00 00 0A 35 A9 59 46 C3 41 41 20 20 20 20 20 20 4E 20 00 00 00 64 4E 43 5A 20 50 4E 20 31 4E 00 00 00 01 4E
          → Stock Directory
   → First Stock Directory! Symbol = 'FAA'

📍 Message  4 | Start byte   96 | Length= 39 | Type='R'
Raw hex → 00 27 52 00 03 00 00 0A 35 A9 59 F7 8D 41 41 41 55 20 20 20 20 50 20 00 00 00 64 4E 51 49 20 50 4E 20 32 59 00 00 00 01 4E
          → Stock Directory

📍 Message  5 | Start byte  137 | Length= 39 | Type='R'
Ra